In [ ]:
import re, os, sys, time
from datetime import datetime
import pandas as pd
from dash import Dash, dash_table, html
import dash_ag_grid as dag
from IPython.display import HTML as html_print
from IPython.display import display
import socket

# repo root so `ebtest.utility.util` (pyfixmsg-backed FixMessage/CODEC) imports
sys.path.insert(0, '/exchbuf/test')

from testplan.testing.multitest.driver.fix import FixClient
from ebtest.utility.util import FixMessage, CODEC, asFIX

from testplan.common.utils.timing import TimeoutException
from uuid import uuid4
from pprint import pprint
from getpass import getuser
import socket
hostname = socket.gethostname().split('.')[0]
user = getuser()
msgs_for_grid = []

def colored(s, color='red'):
    return f'<text style=color:{color}>{s}</text>'

def cprint(s):
    display(html_print(s))

RED = lambda x: cprint(colored(x, 'red'))
GREEN = lambda x: cprint(colored(x, 'green'))
YELLOW = lambda x: cprint(colored(x, 'yellow'))
CYAN = lambda x: cprint(colored(x, 'cyan'))
BLUE = lambda x: cprint(colored(x, 'blue'))
MAGENTA = lambda x: cprint(colored(x, 'magenta'))

def now_ts(ns=True):
    ns_since_epoch = time.time_ns()
    seconds = ns_since_epoch // 1_000_000_000
    dt = datetime.fromtimestamp(seconds)
    time_str = dt.strftime("%Y%m%d-%H:%M:%S") 
    
    if ns:
        ns = ns_since_epoch % 1_000_000_000
        time_str += f".{ns:09d}"
    else:
        ms = ns_since_epoch // 1000 % 1_000_000
        time_str += f".{ms:06d}"
        
    return time_str

def id_generator():
    seq = 0
    prefix = time.strftime('%m%dT%I%M%S')
    while True:
        yield f'OR-{prefix}-{seq}'
        seq += 1

def as_fix(msg):
    if isinstance(msg, str):
        if msg.startswith("50000="):
            msg = msg[msg.find(";") + 1 :]

        if not msg.startswith("8="):
            msg = "8=FIX.4.2;9=0;" + msg

        if (idx := msg.find("Timestamp=")) != -1:
            msg = msg[:idx]

        m = FixMessage()
        m.codec = CODEC
        m.load_fix(msg, separator=';')
        return m
    else:
        return msg

now = lambda: datetime.utcnow().strftime('%Y%m%d-%H:%M:%S.%f')

def fix_apply_modify(msg, modify):
    msg = as_fix(msg)
    for pair in [i.strip() for i in modify.split(';') if i.strip()]:
        try:
            tag, value = pair.split('=')
            tag = int(tag) if tag.isdigit() else tag
            msg.set_or_delete(tag, value)
        except:
            pass
    return msg

def order_summary(msg):
    msg_type = msg.get(35)
    sides = {
        '1': 'buy',
        '2': 'sell',
        '5': 'short sell',
    }

    execType = {
        '0': 'ack',
        '1': 'partial fill',
        '2': 'fill',
        '4': 'canceled',
        '5': 'replace',
        '8': 'rejected'
    }

    get_price = lambda m: 'market' if (m.get(40) == '1') else m.get(44, '')

    clOrdID = msg.get(11)
    origClOrdID = msg.get(41)
    side = sides.get(msg.get(54))
    symbol = msg.get(55, msg.get(48))
    qty = msg.get(38)
    price = get_price(msg)
    exectype = execType.get(msg.get(150), msg.get(150))
    lastShares = msg.get(32)
    lastPx = msg.get(31)
    origQty = ''
    origPrice = ''

    if origClOrdID:
        for m in msgs_for_grid:
            if m.get(11) == origClOrdID:
                origQty = m.get(38, 'n/a')
                origPrice = get_price(m)
    #ack
    for m in msgs_for_grid:
        if m.get(11) == clOrdID:
            if not lastShares:
                lastShares = m.get(38)
            if not lastPx:
                lastPx = get_price(m)
    
    if msg_type == 'D':
        tag10914 = msg.get(10914, '')
        tag14056 = msg.get(14056, '')
        additional = f"10914={tag10914} 14056={tag14056}" if (tag10914 or tag14056) else ''
        BLUE(f'{side} {symbol} {qty}@{price} 11={clOrdID} {additional}')
        
    elif msg_type == 'G':
        BLUE(f'amend: qty:{origQty}=>{qty} price:{origPrice}=>{price} ordID:{clOrdID}=>{origClOrdID}')
        
    elif msg_type == 'F':
        BLUE(f'cancel {origClOrdID}=>{clOrdID}')
        
    elif msg_type == '8':
        if exectype == 'rejected':
            RED(f'{exectype} {side} {symbol} 11={clOrdID} 58={msg.get(58)}')
        else:
            GREEN(f'{exectype} {side} {symbol} {lastShares}@{lastPx} 11={clOrdID}')
            
    elif msg_type == '9':
        RED(f'Cancel Reject 11={clOrdID} 58={msg.get(58)}')
        
    elif msg_type == 'US':
        if msg.get(9604, '') == '1':
            CYAN(f'US {msg.get(9606, "")} available')
        else:
            RED(f'US {msg.get(9606, "")} unavailable')
            
    elif msg_type == 'UCOND':
        BLUE(f'Conditional {side} {symbol} {qty}@{price} 10914={msg.get(10914, "")}')
        
    elif msg_type == 'UCON8':
        if exectype == 'rejected':
            RED(f'{exectype} {side} {symbol} {lastShares}@{lastPx} 10914={msg.get(10914, "")} 27012={msg.get(27012, "")}')
        else:
            GREEN(f'{exectype} {side} {symbol} {lastShares}@{lastPx} 10914={msg.get(10914, "")} 27012={msg.get(27012, "")}')


client = None
client_id = f'fixshell_{hostname}_{user}_{datetime.utcnow().strftime("%f")}'

highlight = '11, 35, 150, 38, 44, 48, 55, 54, 58, 59'
htags = re.split('[, ]+', highlight)
grid_begin_tags = [11, 35, 150, 38, 44, 48, 55, 54, 58, 59, 29, 30, 10484]


def create_client(target, sender='SenderCompID1', target_comp='TargetCompID1'):
    global client
    if client is not None:
        print('client is already created.')
        return

    host, port = target.split(':')

    # Stamp the same custom tags the old MSNet wrapper set on every send.
    class FixClientWrapper(FixClient):
        def send(self, msg):
            msg[7202] = client_id
            msg[52] = now_ts()
            return super().send(msg)

    client = FixClientWrapper(name='fixshell', host=host, port=int(port),
                              sender=sender, target=target_comp,
                              msgclass=FixMessage, codec=CODEC,
                              logon_at_start=False, logoff_at_stop=False)
    client.start()
    client.wait(client.STATUS.STARTED)

    print(f'created client: {client}')


def print_msg(msg, color='red'):
     cprint(';'.join(['='.join((colored(j[0], color), j[1]) if (j:=i.split('=',1))[0] in htags else (j[0], j[1])) for i in msg.output_fix().decode().split(';') if len(i.strip()) != 0]) + ';')

def get_url():
    from IPython.display import display
    from ipyurl import Url
    from jupyter_ui_poll import run_ui_poll_loop

    w = Url()
    display(w)
    url = run_ui_poll_loop(lambda: None if not w.url else w.url)

    return ':'.join(url.split(':')[:-1])

def print_table():
    global msgs_for_grid
    if not msgs_for_grid:
        return

    msgs = msgs_for_grid
        
    all_keys = set(msgs[0].keys())
    for i in msgs[1:]:
        all_keys |= set(i.keys())
        
    begin_tags = [i for i in grid_begin_tags if i in all_keys]

    sorted_keys = begin_tags + sorted(all_keys - set(begin_tags) - {8, 9, 10}) + [8, 9, 10]
    grid = dag.AgGrid(
        id="grid",
        rowData=[{str(k):str(v) for k,v in i.items()} for i in msgs],
        columnDefs=[{"field": str(i)} for i in sorted_keys],
        defaultColDef={"resizable": True, "sortable": False, "filter": False, "minWidth": 125},
        dashGridOptions={"rowSelection": "single", "animateRows": False, "enableCellTextSelection": True, "ensureDomOrder": True},
    )

    #reset msgs_for_grid 
    msgs_for_grid = []
    
    app = Dash('table')
    #port and url for dash server to display grid, when the box is in vLab, the host of url should be ssh proxy host and use this as jupyter_server_url in app.run(..., jupyter_server_url=jupyter_server_url)
    port = 9000
    jupyter_server_url = f'{get_url()}:{port}'
    
    app.layout = html.Div([grid]) 
    app.run(host=socket.gethostname(), port=port, jupyter_server_url=jupyter_server_url)

def get(show_grid=True, timeout=1, show_msg=True):
    while True:
        try:
            msg = client.receive(timeout=timeout)
            global msgs_for_grid
            msgs_for_grid.append(msg)
            if show_msg:
                order_summary(msg)
                print_msg(msg)
        except TimeoutException:
            break

    if show_grid:
        print_table()
        

unique_id = id_generator()

def send(msg, modify=None, show_msg=True, show_grid=True):
    assert client is not None, "please call init_client(...) first"
    msg = as_fix(msg)

    if modify:
        msg = fix_apply_modify(msg, modify)

    if msg[35] == 'D':
        msg[11] = next(unique_id)
    elif msg[35] in ['G', 'F']:
        assert 41 in msg
        msg[11] = msg[41] + msg[35]
        
    msg[52] = now()
    client.send(msg)
    global msgs_for_grid
    msgs_for_grid.append(msg)
    if show_msg:
        order_summary(msg)
        print_msg(msg)
        
    get(show_grid=show_grid)


In [ ]:
create_client('localhost:6666')

In [ ]:
get()

In [ ]:
# define your msg
msg = ''

In [ ]:
# send
send(msg)

In [ ]:
get()